# Tables for DoubleML

[DoubleML](https://docs.doubleml.org/stable/index.html) estimates causal parameters with machine
learning nuisance functions and cross-fitting. Its results objects report one coefficient per
treatment variable, and `maketables` turns them into publication-ready tables like any other model.

This notebook uses the 401(k) eligibility data of Chernozhukov and Hansen (2004): does being
*eligible* for a 401(k) plan (`e401`) raise net financial assets (`net_tfa`)?

In [1]:
# 1. Import dependencies and data
import numpy as np
import pandas as pd
from doubleml import DoubleMLData, DoubleMLIRM, DoubleMLPLR
from doubleml.datasets import fetch_401K
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

import maketables as mt

data = fetch_401K(return_type="DataFrame")
covariates = ["age", "inc", "educ", "fsize", "marr", "twoearn", "db", "pira", "hown"]
dml_data = DoubleMLData(data, y_col="net_tfa", d_cols="e401", x_cols=covariates)

# 2. Fit a partially linear and an interactive regression model
# (the seed fixes DoubleML's cross-fitting split)
np.random.seed(42)
forest = dict(n_estimators=200, max_depth=7, min_samples_leaf=3, random_state=42)

plr = DoubleMLPLR(
    dml_data,
    ml_l=RandomForestRegressor(**forest),
    ml_m=RandomForestClassifier(**forest),
    n_folds=3,
)
plr.fit()

irm = DoubleMLIRM(
    dml_data,
    ml_g=RandomForestRegressor(**forest),
    ml_m=RandomForestClassifier(**forest),
    n_folds=3,
)
irm.fit()

# 3. Table
labels = {
    "net_tfa": "Net financial assets",
    "e401": "401(k) eligibility",
}
mt.ETable(
    [plr, irm],
    labels=labels,
    model_heads=["Partially linear", "Interactive (ATE)"],
    head_order="hd",
    caption="Effect of 401(k) eligibility on net financial assets",
)

<maketables.mtable.MTable.__repr__.<locals>.DualOutput at 0x115232e40>

By default the bottom panel reports the sample size, the score function that identifies the
estimand and the number of cross-fitting folds. Rows for clustering (`n_clusters`) and repeated
cross-fitting (`n_rep`) are added automatically when the model uses them.

`inspect_model` lists everything the extractor can pull out of a given model.

In [2]:
mt.inspect_model(plr)


Model: DoubleMLPLR | Extractor: DoubleMLExtractor

COEFFICIENT TABLE COLUMNS:
  Use these in coef_fmt parameter (e.g., coef_fmt='b:.3f* \n (se:.3f)')
  Available: b, se, t, z, p, ci95l, ci95u, ci90l, ci90u

AVAILABLE STATISTICS:
  Use these in model_stats parameter (e.g., model_stats=['N', 'r2', 'aic'])
  Available: N, se_type, n_folds, n_rep, score, learners
  Defaults: N, score, n_folds

OTHER METADATA:
  depvar=net_tfa, vcov=asymptotic



Any of those columns can be used in `coef_fmt`, and any of the statistics can be requested
explicitly through `model_stats`.

In [3]:
mt.ETable(
    [plr, irm],
    labels=labels,
    coef_fmt="b \n [ci95l, ci95u]",
    model_stats=["N", "score", "n_folds", "n_rep", "learners"],
    model_heads=["Partially linear", "Interactive (ATE)"],
    head_order="hd",
    notes="95% confidence intervals in brackets.",
)

<maketables.mtable.MTable.__repr__.<locals>.DualOutput at 0x115232120>

## Heterogeneous effects

Group and conditional average treatment effects returned by `gate()` and `cate()` are tabled the
same way: one row per group.

In [4]:
groups = pd.DataFrame(
    {"Income tercile": pd.qcut(data["inc"], 3, labels=["Low", "Middle", "High"])}
)
gate = irm.gate(groups)

mt.ETable(
    [gate],
    labels={"y": "Net financial assets"},
    caption="Group average treatment effects by income tercile",
)

<maketables.mtable.MTable.__repr__.<locals>.DualOutput at 0x115232660>

## Notes

- Every estimator that exposes DoubleML's result interface is supported: `DoubleMLPLR`,
  `DoubleMLPLIV`, `DoubleMLIRM`, `DoubleMLIIVM`, `DoubleMLSSM`, `DoubleMLDID`, `DoubleMLAPO`,
  `DoubleMLPQ`, `DoubleMLLPQ`, `DoubleMLCVAR`, as well as `DoubleMLQTE`, `DoubleMLAPOS`,
  `DoubleMLDIDMulti` and the best linear predictors from `cate()`/`gate()`.
- Models with several treatment variables produce one row per treatment; `DoubleMLQTE` produces one
  row per quantile, `DoubleMLAPOS` one per treatment level and `DoubleMLDIDMulti` one per
  group-time combination.
- DoubleML's inference is based on a normal approximation, so the `t` column of its summary is a
  z statistic; it is available under both the `t` and `z` tokens.
- Passing a model that has not been estimated raises an error asking you to call `fit()` first.